In [ ]:
import json
import shutil
from pathlib import Path


ROOT = Path("/benchmark-eeg/5.0_version/result_5/linear_downsample")
SRC_NAME = "Depression_rest_cross_subject_linear_downsample"
DST_NAME = "Depression_rest_wsn_cross_subject_linear_downsample"
SEEDS = [42, 10, 5]
MODEL_NAME = "reve"
RATIO_NAME = "ratio_full"

# Set to True first if you only want to preview what would be copied/updated.
DRY_RUN = False


def rewrite_path(value):
    if isinstance(value, str):
        return value.replace(SRC_NAME, DST_NAME)
    return value


def copy_model_results(src_model_dir: Path, dst_model_dir: Path) -> None:
    if not src_model_dir.exists():
        raise FileNotFoundError(f"Source model dir does not exist: {src_model_dir}")
    dst_model_dir.mkdir(parents=True, exist_ok=True)

    copied = []
    for src_path in sorted(src_model_dir.iterdir()):
        if src_path.suffix == ".pth":
            continue

        dst_path = dst_model_dir / src_path.name
        if DRY_RUN:
            copied.append((src_path, dst_path))
            continue
        if src_path.is_dir():
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
        else:
            shutil.copy2(src_path, dst_path)
        copied.append((src_path, dst_path))
    return copied



def normalize_summary_paths(summary_path: Path) -> dict:
    summary = json.loads(summary_path.read_text())
    for key in ("best_path", "log_path", "output_dir"):
        summary[key] = rewrite_path(summary.get(key))
    if not DRY_RUN:
        summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    return summary


def write_leaderboard(ratio_dir: Path, split: str, rows: list[dict]) -> None:
    metric_key = f"{split}_accuracy"
    title = f"Leaderboard ({split} accuracy)"

    rows.sort(key=lambda row: (-row[metric_key], row["model"]))
    ranked = []
    for rank, row in enumerate(rows, start=1):
        ranked.append(
            {
                "rank": rank,
                "model": row["model"],
                metric_key: row[metric_key],
                "best_ckpt": row["best_ckpt"],
            }
        )

    if not DRY_RUN:
        (ratio_dir / f"leaderboard_{split}.json").write_text(json.dumps(ranked, indent=2) + "\n")
        lines = [title]
        for row in ranked:
            lines.append(
                f"{row['rank']:02d}. {row['model']:<16} "
                f"acc={row[metric_key]:.4f} ckpt={row['best_ckpt']}"
            )
        (ratio_dir / f"leaderboard_{split}.txt").write_text("\n".join(lines) + "\n")
    return ranked


def rebuild_leaderboards(ratio_dir: Path):
    val_rows = []
    test_rows = []
    for model_dir in sorted(path for path in ratio_dir.iterdir() if path.is_dir()):
        summary_path = model_dir / "summary.json"
        if not summary_path.exists():
            continue
        summary = json.loads(summary_path.read_text())
        model = summary.get("model") or model_dir.name
        best_ckpt = rewrite_path(summary.get("best_path"))

        if summary.get("best_val") is not None:
            val_rows.append(
                {
                    "model": model,
                    "val_accuracy": float(summary["best_val"]),
                    "best_ckpt": best_ckpt,
                }
            )
        test_accuracy = (summary.get("test_metrics") or {}).get("accuracy")
        if test_accuracy is not None:
            test_rows.append(
                {
                    "model": model,
                    "test_accuracy": float(test_accuracy),
                    "best_ckpt": best_ckpt,
                }
            )

    val_ranked = write_leaderboard(ratio_dir, "val", val_rows)
    test_ranked = write_leaderboard(ratio_dir, "test", test_rows)
    return val_ranked, test_ranked


def find_model_rank(rows: list[dict], model: str) -> dict | None:
    for row in rows:
        if row["model"] == model:
            return row
    return None


def main() -> None:
    print(f"SRC: {ROOT / SRC_NAME}")
    print(f"DST: {ROOT / DST_NAME}")
    print(f"MODEL: {MODEL_NAME}")
    print(f"DRY_RUN: {DRY_RUN}")

    for seed in SEEDS:
        src_ratio_dir = ROOT / SRC_NAME / f"seed_{seed}_downsample_t40" / RATIO_NAME
        dst_ratio_dir = ROOT / DST_NAME / f"seed_{seed}_downsample_t40" / RATIO_NAME
        src_model_dir = src_ratio_dir / MODEL_NAME
        dst_model_dir = dst_ratio_dir / MODEL_NAME

        missing = []
        if not src_ratio_dir.exists():
            missing.append(f"source ratio dir: {src_ratio_dir}")
        if not dst_ratio_dir.exists():
            missing.append(f"destination ratio dir: {dst_ratio_dir}")
        if not src_model_dir.exists():
            missing.append(f"source model dir: {src_model_dir}")

        if missing:
            print(f"\nseed {seed}")
            print("  SKIP missing:")
            for item in missing:
                print(f"    - {item}")
            continue

        copied = copy_model_results(src_model_dir, dst_model_dir)
        summary = normalize_summary_paths(dst_model_dir / "summary.json")
        val_ranked, test_ranked = rebuild_leaderboards(dst_ratio_dir)
        val_reve = find_model_rank(val_ranked, MODEL_NAME)
        test_reve = find_model_rank(test_ranked, MODEL_NAME)

        print(f"\nseed {seed}")
        print(f"  copied files: {len(copied)}")
        print(f"  reve best_val: {(summary.get('best_val'))}")
        print(f"  reve test_accuracy: {(summary.get('test_metrics') or {}).get('accuracy')}")
        print(f"  val rank: {val_reve['rank'] if val_reve else 'missing'}")
        print(f"  test rank: {test_reve['rank'] if test_reve else 'missing'}")
        print(f"  summary path: {dst_model_dir / 'summary.json'}")


main()
